In [3]:
import openai
import os

openai.api_key = "YOUR_OPENAI_API_KEY"  # Set your API key here


In [ ]:
pip install chromadb

In [4]:
import pickle
import chromadb
import uuid

# Load embeddings
embedding_path = "/content/drive/MyDrive/Major Project/chunked data new/fixed_embedding_data.pkl"
with open(embedding_path, "rb") as f:
    data = pickle.load(f)

texts = data["texts"]
embeddings = data["embeddings"]

# Set up Chroma
chroma_client = chromadb.PersistentClient(path="/content/chroma_db_1")
collection = chroma_client.get_or_create_collection(name="gene_chunks")

# Only add to DB if not already added
if not collection.count():
    ids = [str(uuid.uuid4()) for _ in texts]
    collection.add(documents=texts, embeddings=embeddings.tolist(), ids=ids)


In [5]:
def get_openai_embedding(text):
    response = openai.embeddings.create(
        input=text,
        model="text-embedding-3-small"
    )
    return response.data[0].embedding

def answer_with_openai(question, top_k=5):
    query_embedding = get_openai_embedding(question)

    results = collection.query(query_embeddings=[query_embedding], n_results=top_k)
    context_chunks = results["documents"][0]
    context = "\n\n".join(context_chunks)

    prompt = f"""You are an expert on gene expression in prostate cancer.
Use the following context to answer the question.

Context:
{context}

Question:
{question}

Answer:"""

    response = openai.chat.completions.create(
        model="gpt-3.5-turbo",
        messages=[
            {"role": "system", "content": "You are a helpful biomedical research assistant."},
            {"role": "user", "content": prompt}
        ],
        temperature=0.3,
        max_tokens=400
    )

    return response.choices[0].message.content


In [7]:
while True:
    question = input("Ask a question (or type 'exit' to quit): ")
    if question.lower() == "exit":
        print("👋 Done!")
        break
    try:
        answer = answer_with_openai(question)
        print(f"\n💬 Answer:\n{answer}\n")
    except Exception as e:
        print(f"⚠️ Error: {e}")

Ask a question (or type 'exit' to quit): exit
👋 Done!
